# Day 57 · Exercise 2: Health Check Endpoint

**What you'll build:** Implement `build_health_api(version)` — a FastAPI app with a `GET /health` endpoint returning status, timestamp, and version. Every production service needs a health endpoint so load balancers can verify the instance is alive.

## Setup (provided)

In [ ]:
from datetime import datetime
from fastapi import FastAPI
from starlette.testclient import TestClient


## Your Implementation

In [ ]:
def build_health_api(version: str = "1.0.0") -> FastAPI:
    """Build a FastAPI app with a GET /health endpoint.

    The /health endpoint must return a JSON object with exactly these keys:
      - "status":    the string "ok"
      - "timestamp": current UTC time as an ISO-8601 string (datetime.utcnow().isoformat())
      - "version":   the version string passed to build_health_api()

    Args:
        version: Application version string (e.g. "1.0.0").
    Returns:
        A FastAPI app instance with the /health route registered.
    """
    app = FastAPI()

    # TODO: add GET /health that returns {"status": "ok", "timestamp": ..., "version": ...}

    return app


In [ ]:
def build_health_api(version: str = "1.0.0") -> FastAPI:
    app = FastAPI()

    @app.get("/health")
    def health():
        return {
            "status": "ok",
            "timestamp": datetime.utcnow().isoformat(),
            "version": version,
        }

    return app


## Check Your Work

In [ ]:
def _run_checks():
    score = 0
    total = 5

    def _chk(n, ok, msg):
        nonlocal score
        print(f"  {'✅' if ok else '❌'} Check {n}: {msg}")
        if ok:
            score += 1

    try:
        app = build_health_api("2.5.0")
    except NotImplementedError:
        for i in range(1, total + 1):
            print(f"  ❌ Check {i}: build_health_api not implemented")
        print(f"\nScore: 0 / {total}")
        return
    except Exception as e:
        for i in range(1, total + 1):
            print(f"  ❌ Check {i}: {type(e).__name__}: {e}")
        print(f"\nScore: 0 / {total}")
        return

    client = TestClient(app, raise_server_exceptions=False)
    r = client.get("/health")

    _chk(1, r.status_code == 200,
         f"GET /health → 200 (got {r.status_code})")

    if r.status_code == 200:
        data = r.json()
        _chk(2, data.get("status") == "ok",
             f"status == 'ok' (got {data.get('status')!r})")
        _chk(3, isinstance(data.get("timestamp"), str) and "T" in data.get("timestamp", ""),
             f"timestamp is ISO-8601 string (got {data.get('timestamp')!r})")
        _chk(4, data.get("version") == "2.5.0",
             f"version == '2.5.0' (got {data.get('version')!r})")
    else:
        for i in range(2, 5):
            print(f"  ❌ Check {i}: skipped (check 1 failed)")

    # different version
    app2 = build_health_api("0.1.0")
    r2 = TestClient(app2, raise_server_exceptions=False).get("/health")
    _chk(5, r2.status_code == 200 and r2.json().get("version") == "0.1.0",
         f"version from second app == '0.1.0' (got {r2.json().get('version') if r2.status_code == 200 else r2.status_code!r})")

    print(f"\nScore: {score} / {total}")
    if score == total:
        print("🎉 Exercise complete!")

_run_checks()


## Bonus Challenge

Extend `/health` to include a `'checks'` list that verifies sub-dependencies. For example, add a `check_ollama()` helper that tries `ollama.list()` — if it raises, include `{'ollama': 'unreachable'}` in the checks list and set overall `'status': 'degraded'` instead of 'ok'. This is a deep health check (vs a shallow ping).

## Solution

<details>
<summary>Show solution</summary>

```python
def build_health_api(version: str = "1.0.0") -> FastAPI:
    app = FastAPI()

    @app.get("/health")
    def health():
        return {
            "status": "ok",
            "timestamp": datetime.utcnow().isoformat(),
            "version": version,
        }

    return app
```

**Why this works:** The `/health` endpoint is intentionally minimal — it has no
dependencies, no database calls, no Ollama calls. It just returns three fields.
Load balancers and deployment platforms call `/health` on a fixed interval; if
it returns 200 the instance is healthy, if it times out or returns 5xx the
instance is removed. `datetime.utcnow().isoformat()` gives a UTC timestamp in
ISO-8601 format (`2026-07-25T14:30:00.123456`) which is unambiguous across
time zones. The version field helps identify which code version is deployed.

</details>